# Lab 2 — NumPy Arrays & Vectorized Math

**Day 02 · Python for Data Science · Cisco AI/ML Training**

---

## Goals

1. Build **1-D and 2-D** `ndarray` objects from Python lists and CSV columns.
2. Apply **vectorized** math instead of Python loops.
3. **Normalize** features with mean and standard deviation (z-score).
4. Use **boolean masks**, **axis** reductions, and basic **linear algebra** on feature vectors.

> **Quick check:** `votes.shape == (5,)`, `matrix.shape == (5, 2)`, column means ≈ **[1392, 1290]**.




## Why NumPy underpins data science

| Without NumPy | With NumPy |
|---------------|------------|
| Slow loops over thousands of rows | C-optimized vectorized kernels |
| Lists with mixed types | Homogeneous `dtype` (`int64`, `float64`) |
| Manual linear algebra | `dot`, broadcasting, `axis` reductions |

Pandas Series and scikit-learn estimators sit on NumPy. Today we use **vote counts**, **meal costs**, and **ratings** from Zomato — the same patterns scale to all **500** rows in Lab 3.

## Vector space (course topic)

Each restaurant is a **vector** in ℝⁿ (one number per feature). Day 4 distance metrics reuse the same idea on loan feature vectors.

| Concept | NumPy | Day 4 link |
|---------|-------|------------|
| Vector | 1-D `ndarray` | Loan feature row |
| Magnitude | `np.linalg.norm(v)` | Euclidean distance |
| Direction | unit vector `v / norm(v)` | Cosine similarity |

## NumPy cheat sheet (this lab)

| Task | Code |
|------|------|
| Create from list | `np.array([1, 2, 3])` |
| Evenly spaced | `np.arange(0, 10, 2)` |
| Z-score | `(x - x.mean()) / x.std()` |
| Filter | `x[x > threshold]` |
| Column matrix | `np.column_stack([a, b])` |
| Mean per column | `matrix.mean(axis=0)` |

---

## 0. Load Zomato numeric columns into NumPy

In [1]:
from pathlib import Path
import csv
import time

import numpy as np

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

ZOMATO_CSV = GH_ROOT / "data" / "zomato" / "zomato_restaurants.csv"

**Step 2** — run the cell below and read the printed summary. <!-- cisco-split-debug-2026 -->

In [2]:
def load_zomato_columns(path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    ratings, votes, costs = [], [], []
    with path.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            ratings.append(float(row["aggregate_rating"]))
            votes.append(int(row["votes"]))
            costs.append(int(row["average_cost_for_two"]))
    return np.array(ratings), np.array(votes), np.array(costs)

**Step 3** — run the cell below and read the printed summary. <!-- cisco-split-debug-2026 -->

In [3]:
all_ratings, all_votes, all_costs = load_zomato_columns(ZOMATO_CSV)
print("Loaded rows:", len(all_ratings))
print("ratings  → shape", all_ratings.shape, "dtype", all_ratings.dtype)
print("votes    → shape", all_votes.shape)
print("costs    → shape", all_costs.shape)
print("Sample ratings:", np.round(all_ratings[:6], 2))

Loaded rows: 500
ratings  → shape (500,) dtype float64
votes    → shape (500,)
costs    → shape (500,)
Sample ratings: [3.7 2.7 2.7 3.  3.1 3.1]


---

## 1. Checkpoint arrays — five sample restaurants

We keep a **small fixed sample** for locked assertions; the full **500**-row arrays above are for practice at scale.

In [4]:
votes = np.array([120, 450, 890, 2100, 3400])
costs = np.array([400, 650, 1200, 1800, 2400])

print("votes:", votes)
print("costs:", costs)
print("votes shape:", votes.shape, "| dtype:", votes.dtype)
print("costs shape:", costs.shape)


votes: [ 120  450  890 2100 3400]
costs: [ 400  650 1200 1800 2400]
votes shape: (5,) | dtype: int64
costs shape: (5,)


### 1b. Creating arrays — `arange`, `linspace`, `zeros`

In [5]:
# Price tiers for a promo banner (₹ steps)
price_steps = np.arange(200, 1001, 200)
print("arange:", price_steps)

# Five evenly spaced rating thresholds for bucketing
rating_bins = np.linspace(2.5, 4.5, 5)
print("linspace:", np.round(rating_bins, 2))

# Placeholder feature matrix (3 restaurants × 2 features) — filled later
placeholder = np.zeros((3, 2))
print("zeros:\n", placeholder)


arange: [ 200  400  600  800 1000]
linspace: [2.5 3.  3.5 4.  4.5]
zeros:
 [[0. 0.]
 [0. 0.]
 [0. 0.]]


### 1c. `shape`, `ndim`, and `size`

In [6]:
print("votes.ndim:", votes.ndim, "| size:", votes.size)
matrix_preview = np.column_stack([votes, costs])
print("2-D preview shape:", matrix_preview.shape, "ndim:", matrix_preview.ndim)


votes.ndim: 1 | size: 5
2-D preview shape: (5, 2) ndim: 2


---

## 2. Vectorized arithmetic

In [7]:
revenue_proxy = costs * 1.1  # hypothetical 10% markup
print("Markup revenue (first 3):", np.round(revenue_proxy[:3], 2))

cost_per_vote = costs / votes
print("cost_per_vote (first 3):", np.round(cost_per_vote[:3], 3))


Markup revenue (first 3): [ 440.  715. 1320.]
cost_per_vote (first 3): [3.333 1.444 1.348]


### 2b. Broadcasting — scalar operates on every element

In [8]:
adjusted_ratings = all_ratings + 0.0  # copy-like op on full dataset
print("Mean rating (500 rows):", round(adjusted_ratings.mean(), 2))

# Clip unrealistic ratings if any data entry error (cap at 5.0)
clipped = np.minimum(all_ratings, 5.0)
print("Any values clipped?", not np.array_equal(clipped, all_ratings))


Mean rating (500 rows): 3.7
Any values clipped? False


### 2c. Loop vs vectorized (500 rows)

In [9]:
# Slow pattern
t0 = time.perf_counter()
loop_result = []
for v, c in zip(all_votes[:500], all_costs[:500]):
    loop_result.append(c / v)
loop_ms = (time.perf_counter() - t0) * 1000

# Fast pattern
t0 = time.perf_counter()
vec_result = all_costs[:500] / all_votes[:500]
vec_ms = (time.perf_counter() - t0) * 1000

print(f"Loop: {loop_ms:.2f} ms | Vectorized: {vec_ms:.3f} ms")
print("Same results:", np.allclose(loop_result, vec_result))


Loop: 0.28 ms | Vectorized: 0.116 ms
Same results: True


---

## 3. Normalization (z-score)

Standardization: \(z = (x - \mu) / \sigma\). Used before KNN (Day 4) and inside `StandardScaler` (Day 3+).

In [10]:
mean_votes = votes.mean()
std_votes = votes.std()
print(f"mean votes: {mean_votes}")
print(f"std votes:  {std_votes}")

normalized_votes = (votes - votes.mean()) / votes.std()
print("normalized_votes:", np.round(normalized_votes, 3))
print("normalized mean (~0):", np.round(normalized_votes.mean(), 6))


mean votes: 1392.0
std votes:  1207.5330223227852
normalized_votes: [-1.053 -0.78  -0.416  0.586  1.663]
normalized mean (~0): 0.0


### 3b. Z-score on full Zomato ratings

In [11]:
z_ratings = (all_ratings - all_ratings.mean()) / all_ratings.std()
print("Full-dataset rating z-scores — min/max:", np.round(z_ratings.min(), 2), np.round(z_ratings.max(), 2))
print("Share above +1 std:", round((z_ratings > 1).mean() * 100, 1), "%")


Full-dataset rating z-scores — min/max: -1.69 1.69
Share above +1 std: 20.4 %


### 3c. Percentiles and spread

In [12]:
print("Rating percentiles (500 rows):")
for p in (25, 50, 75, 90):
    print(f"  P{p}:", round(np.percentile(all_ratings, p), 2))
print("Std dev ratings:", round(all_ratings.std(), 3))


Rating percentiles (500 rows):
  P25: 3.1
  P50: 3.7
  P75: 4.4
  P90: 4.7
Std dev ratings: 0.71


---

## 4. Indexing, slicing, and boolean masks

In [13]:
print("votes[1:4]:", votes[1:4])
print("votes[-1]:", votes[-1])

high_engagement = votes > 1000
print("mask:", high_engagement)
print("costs where votes > 1000:", costs[high_engagement])


votes[1:4]: [ 450  890 2100]
votes[-1]: 3400
mask: [False False False  True  True]
costs where votes > 1000: [1800 2400]


### 4b. Mask on real data — highly rated restaurants

In [14]:
top_mask = all_ratings >= 4.5
print("Count rated ≥ 4.5:", top_mask.sum())
print("Mean votes (top rated):", round(all_votes[top_mask].mean(), 0))
print("Mean votes (others):", round(all_votes[~top_mask].mean(), 0))


Count rated ≥ 4.5: 102
Mean votes (top rated): 2511.0
Mean votes (others): 2504.0


### 4c. `np.where` — conditional values

In [15]:
engagement_label = np.where(all_votes >= 3000, "high", "moderate")
unique, counts = np.unique(engagement_label, return_counts=True)
print(dict(zip(unique, counts)))


{np.str_('high'): np.int64(202), np.str_('moderate'): np.int64(298)}


### 4d. Aggregations on 500-row arrays

In [16]:
print("Votes — min:", all_votes.min(), "max:", all_votes.max(), "sum:", all_votes.sum())
print("Costs — mean:", round(all_costs.mean(), 0), "median:", np.median(all_costs))
print("Ratings — mean:", round(all_ratings.mean(), 2))


Votes — min: 12 max: 4976 sum: 1252539
Costs — mean: 1294.0 median: 1320.5
Ratings — mean: 3.7


### 4e. Histogram bins (rating distribution preview)

In [17]:
counts, bin_edges = np.histogram(all_ratings, bins=[2.5, 3.0, 3.5, 4.0, 4.5, 5.1])
for i in range(len(counts)):
    lo, hi = bin_edges[i], bin_edges[i + 1]
    print(f"  [{lo:.1f}, {hi:.1f}): {counts[i]} restaurants")


  [2.5, 3.0): 96 restaurants
  [3.0, 3.5): 100 restaurants
  [3.5, 4.0): 106 restaurants
  [4.0, 4.5): 96 restaurants
  [4.5, 5.1): 102 restaurants


---

## 5. Two-dimensional feature matrix

In [18]:
matrix = np.column_stack([votes, costs])
print("matrix:\n", matrix)
print("matrix shape:", matrix.shape)


matrix:


 [[ 120  400]
 [ 450  650]
 [ 890 1200]
 [2100 1800]
 [3400 2400]]
matrix shape: (5, 2)


### 5b. Axis reductions

| Expression | Meaning |
|------------|--------|
| `matrix.mean(axis=0)` | Mean **per column** |
| `matrix.mean(axis=1)` | Mean **per row** |
| `matrix.sum(axis=0)` | Sum per column |

In [19]:
col_means = matrix.mean(axis=0)
row_means = matrix.mean(axis=1)

print("column means [votes, cost]:", np.round(col_means, 2))
print("row means:", np.round(row_means, 2))
print("sum axis=0:", matrix.sum(axis=0))


column means [votes, cost]: [1392. 1290.]
row means: [ 260.  550. 1045. 1950. 2900.]
sum axis=0: [6960 6450]


### 5c. Build a 50×3 matrix from Zomato (rating, votes, cost)

In [20]:
sample_n = 50
feature_block = np.column_stack([
    all_ratings[:sample_n],
    all_votes[:sample_n],
    all_costs[:sample_n],
])
print("feature_block shape:", feature_block.shape)
print("Column means:", np.round(feature_block.mean(axis=0), 2))
print("Per-restaurant avg feature (first 3 rows):", np.round(feature_block.mean(axis=1)[:3], 2))


feature_block shape: (50, 3)
Column means: [   3.69 2375.9  1350.38]
Per-restaurant avg feature (first 3 rows): [1351.23 1066.23 1654.23]


### 5d. `reshape` — column vector for sklearn-style inputs

In [21]:
votes_col = votes.reshape(-1, 1)
print("votes_col shape:", votes_col.shape)
print(votes_col)


votes_col shape: (5, 1)
[[ 120]
 [ 450]
 [ 890]
 [2100]
 [3400]]


### 5e. `vstack` and `concatenate`

In [22]:
row_a = np.array([1, 2, 3])
row_b = np.array([4, 5, 6])
stacked = np.vstack([row_a, row_b])
print("vstack shape:", stacked.shape)
print(stacked)

# concatenate 1-D arrays end-to-end
combined = np.concatenate([votes[:2], costs[:2]])
print("concatenate [votes[:2], costs[:2]]:", combined)


vstack shape: (2, 3)
[[1 2 3]
 [4 5 6]]
concatenate [votes[:2], costs[:2]]: [120 450 400 650]


### 5f. Transpose and row/column views

In [23]:
print("matrix.T (swap rows/cols):\n", matrix.T)
print("Row 2 features:", matrix[2])
print("Column 0 (all votes):", matrix[:, 0])


matrix.T (swap rows/cols):
 [[ 120  450  890 2100 3400]
 [ 400  650 1200 1800 2400]]
Row 2 features: [ 890 1200]
Column 0 (all votes): [ 120  450  890 2100 3400]


---

## 6. Linear algebra on feature vectors

In [24]:
v = np.array([120.0, 400.0])
w = np.array([450.0, 650.0])
print("dot product v·w:", np.dot(v, w))
print("norm(v):", round(np.linalg.norm(v), 2))
print("norm(w):", round(np.linalg.norm(w), 2))


dot product v·w: 314000.0
norm(v): 417.61
norm(w): 790.57


### 6b. Unit vector and cosine setup

In [25]:
def unit_vector(x: np.ndarray) -> np.ndarray:
    return x / np.linalg.norm(x)

u = unit_vector(v.astype(float))
print("unit vector:", np.round(u, 4))
print("unit length:", round(np.linalg.norm(u), 6))


unit vector: [0.2873 0.9578]
unit length: 1.0


### 6c. Correlation — votes vs ratings (500 rows)

In [26]:
corr = np.corrcoef(all_votes, all_ratings)[0, 1]
print("Pearson corr(votes, rating):", round(corr, 3))


Pearson corr(votes, rating): 0.001


### 6d. `argmax` — index of busiest outlet in sample

In [27]:
busiest_idx = int(np.argmax(votes))
print("Busiest sample index:", busiest_idx)
print("Votes:", votes[busiest_idx], "| Cost:", costs[busiest_idx])


Busiest sample index: 4
Votes: 3400 | Cost: 2400


### 6e. Euclidean distance between two restaurants (2 features)

In [28]:
def euclidean(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b))

r0 = matrix[0]  # votes, cost for restaurant 0
r1 = matrix[1]
print("Distance (restaurant 0 vs 1):", round(euclidean(r0, r1), 2))
print("(Day 4 KNN generalizes this to many features and neighbors)")


Distance (restaurant 0 vs 1): 414.0
(Day 4 KNN generalizes this to many features and neighbors)


### 6f. Element-wise vs matrix multiply

In [29]:
ew = votes * costs
print("Element-wise votes * costs:", ew)
# Matrix multiply needs 2-D conformable shapes — not used on two 1-D vectors directly
print("For 1-D vectors use np.dot for inner product, * for element-wise.")


Element-wise votes * costs:

 [  48000  292500 1068000 3780000 8160000]
For 1-D vectors use np.dot for inner product, * for element-wise.


---

## 7. Experiment — change one vote

In [30]:
# Uncomment to explore sensitivity:
# votes_experiment = votes.copy()
# votes_experiment[2] = 5000
# print((votes_experiment - votes_experiment.mean()) / votes_experiment.std())

print("Checkpoint uses original votes:", votes.tolist())


Checkpoint uses original votes: [120, 450, 890, 2100, 3400]


---

## 8. Try it yourself

Using `all_ratings` and `all_costs`:
1. Compute **median** cost with `np.median`.
2. Build a mask for restaurants with **cost < median** and **rating ≥ 4.0**.
3. Print how many match.

In [31]:
# Your code (optional)


In [32]:
median_cost = np.median(all_costs)
premium_value = (all_costs < median_cost) & (all_ratings >= 4.0)
print("Median cost for two:", int(median_cost))
print("High rating + below-median cost:", premium_value.sum())


Median cost for two: 1320
High rating + below-median cost: 95


---

## 9. Final checkpoint

In [33]:
print("Lab 2 — NumPy arrays")
print(f"votes shape: {votes.shape}, dtype: {votes.dtype}")
print(f"normalized_votes: {np.round(normalized_votes, 3)}")
print(f"cost_per_vote (first 3): {np.round(cost_per_vote[:3], 3)}")
print(f"matrix shape: {matrix.shape}")
print(f"column means [votes, cost]: {np.round(col_means, 2)}")

assert votes.shape == (5,)
assert matrix.shape == (5, 2)
assert abs(col_means[0] - 1392) < 1 and abs(col_means[1] - 1290) < 1
print("\n✓ Checkpoint assertions passed")


Lab 2 — NumPy arrays
votes shape: (5,), dtype: int64
normalized_votes: [-1.053 -0.78  -0.416  0.586  1.663]
cost_per_vote (first 3): [3.333 1.444 1.348]
matrix shape: (5, 2)
column means [votes, cost]: [1392. 1290.]

✓ Checkpoint assertions passed


---

## Reflection

1. Why is `votes.shape` displayed as `(5,)`?
2. When would `axis=1` matter for a Zomato feature matrix?
3. How does `normalized_votes` relate to `StandardScaler`?
4. Why is vectorized division faster than a Python loop on 500 rows?
